# NatureInsight® Scenario Sensitivity and Summary Extraction

This notebook processes NatureInsight® / SCALGO scenario exports for the River Wansbeck dissertation analysis. It extracts headline metrics, calculates derived fields, and produces a clean master summary table for threshold, ranking, and scenario comparison. With simple adjustments, this can cater for catchment downloads from the SCALGO LIVE platform. Note, they must have a noted Scenario ID, for ease, i saved files under matching names.


## Notebook workflow

1. Import analysis libraries.
2. Define the NatureInsight® export labels required for extraction.
3. Parse scenario IDs into rank, suitability threshold, and catchment metadata.
4. Extract scenario metrics from uploaded Excel exports.
5. Calculate derived comparison fields, including percentage catchment coverage and percentage change from baseline.
6. Export a master summary table for results and discussion.

**Expected scenario ID examples:** `R1-T70`, `R2-T70`, `R1-SS70-WAN`, `R1-SS40-CAL`.

## 1. Import libraries

These libraries support spreadsheet handling, file upload within Google Colab, regular expression parsing, and numerical processing.

In [ ]:
import pandas as pd
from google.colab import files
from openpyxl import load_workbook
import io
import re
import numpy as np


## 2. Define NatureInsight® export labels

The NatureInsight® / SCALGO exports contain named metric rows. This dictionary maps those row labels onto clearer column names for the summary table.

Standardised tables improve consistency between scenarios, automated table generation and analysis.

In [ ]:
ROW_LABELS = {
    "Number of opportunities": "Total Outputs", # Total number of intervention opportunities identified
    "Storage": "Total Storage (m³)",
    "Area": "Total Area (m²)", # Total intervention area
    "Net carbon sequestration": "Total Net Carbon", # Estimated net carbon sequestration
    "Net habitat units": "Total Net Habitat",  # Estimated biodiversity gain
    "Cost": "Total Cost (£)" # Estimated implementation cost, uses Spon's.
}

# Column K contains the total values in the NatureInsight® / SCALGO export.
TOTAL_COLUMN_INDEX = 10


## 3. Parse scenario IDs

Scenario IDs are converted into structured metadata so that outputs can be sorted and compared consistently. The parser accepts both the earlier `R1-SS70` style and the later `R1-SS70-WAN` style. For my study, I had a supporting table which defined each, ie R1 is rank 1. These would be adjustable for the purpose of alternative studies. This was to provide consitnecy in labelling to both simplify the code inputs but also to increase readability and referencing through my research.

In [ ]:
# Components:
# R1     = rank selection
# SS40   = suitability score threshold
# CAL    = catchment identifier
# identifiers also for storage bucket type, order by, and storage parameters.

def parse_scenario_id(scenario_id):
    # Preferred format: R1-SS40-CAL
    # addition represetns the catchment, so code can be used for multiple catchment comparisons.
    pattern_new = r"R(\d+)-SS(\d+)(?:-([A-Z]+))?"
    match_new = re.fullmatch(pattern_new, scenario_id.strip().upper())

    if match_new:
        rank = int(match_new.group(1)) # Extract rank value
        threshold = int(match_new.group(2))# Extract SS value
        catchment = match_new.group(3) if match_new.group(3) else "WAN"  # default to Wansbeck, as this is my baseline study focus.
        return rank, threshold, catchment

    # Earlier format retained for compatibility: R1-T70
    pattern_old = r"R(\d+)-T(\d+)"
    match_old = re.fullmatch(pattern_old, scenario_id.strip().upper())

    if match_old:
        rank = int(match_old.group(1))
        threshold = int(match_old.group(2))
        return rank, threshold, "WAN"
 # Invalid scenario naming, for incorrect parsing and lebels.
    raise ValueError(f"Scenario ID '{scenario_id}' must look like R1-T70 or R1-SS40-CAL")

## 4. Extract metrics from one export

This function reads one uploaded Excel export and returns a single summary row. The `openpyxl` method is used because it handles formatted Excel exports more reliably than a direct `pandas.read_excel()` approach for this workflow.

This section also idetnifies cominant intervention types and standardises outputs. The output seeks to support scneario comparison, sensitivity analysis, plotting, data abstraction and summarisation.

In [ ]:
def extract_scalgo_metrics(file_bytes, scenario_id):
    wb = load_workbook(io.BytesIO(file_bytes), data_only=True) # Open excel

    # Print sheet names as a quick workbook check.
    print("Available sheets:", wb.sheetnames)

    # NatureInsight® exports are expected to use the first worksheet.
    ws = wb[wb.sheetnames[0]]

     # Parse scenario metadata for clarity
     rank, threshold, catchment = parse_scenario_id(scenario_id)

 # Stores extracted metrics in a standardised format for later dataframe creation and analysis and clarity trhroughout.
    row_data = {
        "Scenario ID": scenario_id,
        "Rank": rank,
        "Threshold": threshold,
        "Catchment": catchment,
        "Total Outputs": None,
        "Total Area (m²)": None,
        "% Catchment Covered": None,
        "Total Storage (m³)": None,
        "Total Net Carbon": None,
        "Total Net Habitat": None,
        "Total Cost (£)": None,
        "Dominant Intervention": None,
        "% Change Outputs (vs baseline)": None,
        "% Change Storage": None,
        "% Change Cost": None,
        "Notes": None
    }

    # Extract total summary metrics, Read labels in column A and totals in column K.
    for row in range(1, ws.max_row + 1):
        label = ws.cell(row=row, column=1).value

        if label in ROW_LABELS:
            mapped_name = ROW_LABELS[label]
            total_value = ws.cell(row=row, column=11).value  # Column K
            # Store extracted values

            if mapped_name == "Total Outputs":
                row_data["Total Outputs"] = total_value
            elif mapped_name == "Total Area (m²)":
                row_data["Total Area (m²)"] = total_value
            elif mapped_name == "Total Storage (m³)":
                row_data["Total Storage (m³)"] = total_value
            elif mapped_name == "Total Net Carbon":
                row_data["Total Net Carbon"] = total_value
            elif mapped_name == "Total Net Habitat":
                row_data["Total Net Habitat"] = total_value
            elif mapped_name == "Total Cost (£)":
                row_data["Total Cost (£)"] = total_value

    # Identify the intervention with the largest number mapped of opportunities.
    intervention_names = []
    for col in range(2, 11):  # B to J
        intervention_names.append(ws.cell(row=1, column=col).value)

    number_row = None # Locate the "Number of opportunities" row
    for row in range(1, ws.max_row + 1):
        if ws.cell(row=row, column=1).value == "Number of opportunities":
            number_row = row
            break

    # Extract intervention counts
    if number_row is not None:
        counts = []
        for col in range(2, 11):
            val = ws.cell(row=number_row, column=col).value
            counts.append(val if val is not None else 0)

        # Store the intervention with the largest count.
        max_idx = counts.index(max(counts))
        row_data["Dominant Intervention"] = intervention_names[max_idx]

    return row_data

## 5. Calculate derived fields

This function standardises numeric columns and adds additional analysis fields, including percentage catchment coverage and percentage change from a selected baseline scenario. **Note to adapt these baselines accordingly, the data can be obtained from within SCALGOLIVE.**

These derived fields are used to support scenario comparison,
including:
- catchment coverage
- percentage change in outputs
- percentage change in storage
- percentage change in cost

In [ ]:
def add_derived_fields(summary_df, catchment_area=None, baseline_id="R1-T70"):

    # Convert relevant columns to numeric values before calculation.

    # Excel exports can sometimes import values as text.
    # Converting these columns ensures calculations work correctly and prevents errors during percentage change calculations.

    numeric_cols = [
        "Total Outputs",
        "Total Area (m²)",
        "Total Storage (m³)",
        "Total Net Carbon",
        "Total Net Habitat",
        "Total Cost (£)"
    ]

    for col in numeric_cols:
        summary_df[col] = pd.to_numeric(summary_df[col], errors="coerce")

    # Calculate catchment coverage where catchment area is supplied.
    # This is only calculated where a catchment area is provided by the user. Please insert one if comparisons wanted, will e blank in table if not.
    # Formula:
    # intervention area / total catchment area × 100
    if catchment_area is not None and catchment_area != 0:
        summary_df["% Catchment Covered"] = (
            summary_df["Total Area (m²)"] / float(catchment_area) * 100
        )

    # Compare scenario outputs against the selected baseline scenario. This provides a reference for other scenario comparisons, I used R1-SS70 as my default, due to natureInsight tool using this as the default.
    baseline_rows = summary_df[summary_df["Scenario ID"] == baseline_id]

     # Percentage change is only calculated where:
    # - the baseline exists
    # - baseline values are not missing
    # - baseline values are not zero

    # This avoids divide-by-zero errors and misleading outputs.

    if not baseline_rows.empty:
        baseline_outputs = baseline_rows.iloc[0]["Total Outputs"]
        baseline_storage = baseline_rows.iloc[0]["Total Storage (m³)"]
        baseline_cost = baseline_rows.iloc[0]["Total Cost (£)"]

        # Percentage change in number of mapped outputs
        if pd.notna(baseline_outputs) and baseline_outputs != 0:
            summary_df["% Change Outputs (vs baseline)"] = (
                (summary_df["Total Outputs"] - baseline_outputs) / baseline_outputs * 100
            )


        # Percentage change in total estimated storage
        if pd.notna(baseline_storage) and baseline_storage != 0:
            summary_df["% Change Storage"] = (
                (summary_df["Total Storage (m³)"] - baseline_storage) / baseline_storage * 100
            )

        # Percentage change in estimated cost
        if pd.notna(baseline_cost) and baseline_cost != 0:
            summary_df["% Change Cost"] = (
                (summary_df["Total Cost (£)"] - baseline_cost) / baseline_cost * 100
            )

    return summary_df # Rerutn adjusted dataframe.

## 6. Upload multiple scenario exports

Upload all NatureInsight® / SCALGO exports together. The filenames are sorted alphabetically so that the scenario ID matching step is easier to check. No limit on uploaded files here. I would recomend to upload all the fils you would like to summarise, however, there is an add on feature built-in to this code later.

In [ ]:
uploaded = files.upload()

# Sort uploaded files alphabetically so scenario matching is transparent.
# This helps reduce confusion when matching uploaded files
# to scenario IDs.
uploaded_filenames = sorted(uploaded.keys())

print("Files uploaded in this order:")
# Display uploaded file order

# Printing the order makes it easier to check that scenario IDs are assigned to the correct files.
for i, name in enumerate(uploaded_filenames, start=1):
    print(f"{i}. {name}")

## 7. Enter matching scenario IDs

Enter scenario IDs in the same order as the printed file list. This step links each uploaded export to its rank and threshold information.

In [ ]:
# Example:
# If the files were printed as:
# 1. export_A.xlsx
# 2. export_B.xlsx
# Then the first scenario ID entered will be assigned to
# export_A.xlsx, and the second to export_B.xlsx.scenario_input = input(
    "Enter Scenario IDs in the SAME ORDER, comma-separated:\n"
    "Example: R1-ss70, R2-ss70, R3-ss70\n"
).strip()

# Split the comma-separated text into a list of scenario IDs
scenario_ids = [x.strip() for x in scenario_input.split(",")]
# Check number of scenario IDs matches number of files

if len(scenario_ids) != len(uploaded_filenames):
    raise ValueError(
        f"You entered {len(scenario_ids)} Scenario IDs, "
        f"but uploaded {len(uploaded_filenames)} files."
    )
# Display file-to-scenario mapping
print("\nScenario mapping:")  # this shows if mapped correctly, so check here to ensure you entered the correct order.
for file_name, scenario_id in zip(uploaded_filenames, scenario_ids):
    print(f"{file_name}  -->  {scenario_id}")

## 8. Build the master summary table

Each uploaded export is processed into one row. The combined table is then sorted by threshold and rank so that the results are ready for interpretation.

Each extracted row represents one scenario.

In [ ]:
all_rows = []

for file_name, scenario_id in zip(uploaded_filenames, scenario_ids):
    file_bytes = uploaded[file_name] # Read uploaded Excel file as bytes
    print(f"\nProcessing {file_name} as {scenario_id}")  # Print progress so processing can be checked
    row = extract_scalgo_metrics(file_bytes, scenario_id)  # Extract scenario metrics using the custom extraction function
    all_rows.append(row)  # Add extracted row to results list
 # Make a summary of data.
summary_df = pd.DataFrame(all_rows)
 # Whereby catchment area is given, the notebook can calulcate the precentage of the ctchment covered by mapped interventions.
catchment_input = input(
    "\nEnter total catchment area in m² for % cover calculation (or leave blank): "
).strip()

catchment_area = float(catchment_input) if catchment_input else None

summary_df = add_derived_fields(summary_df, catchment_area=catchment_area, baseline_id="R1-T70")
# Adds calculated metrics including:
# - percentage catchment coverage
# - percentage change in outputs
# - percentage change in storage
# - percentage change in cost
# The baseline scenario can be changed if required/wanted.
summary_df = summary_df.sort_values(by=["Threshold", "Rank"]).reset_index(drop=True)
# Sorting by threshold and rank helps make scenario outputs easier to compare in tables and plots
summary_df # Display final summary table

## 9. Save the master summary table

The final table is exported as an Excel workbook for use in the dissertation results, appendices, or further plotting.

In [ ]:
output_name = "scalgo_master_summary.xlsx"
summary_df.to_excel(output_name, index=False)

print(f"Saved file: {output_name}")

files.download(output_name)

## 10. Optional single-export checker

Use this section when you only need to process one scenario export and generate a paste-ready row for manual Excel entry. E.g at a later date, you wanted to add an additional scenario.

This was added to encourage iterations to studies, and to develop outputs depending on results found. I.e if wanting to increase the scope of ranks trialed etc.

In [ ]:
# Upload a single NatureInsight® / SCALGO export
uploaded = files.upload()

file_name = list(uploaded.keys())[0]
file_bytes = uploaded[file_name]

# Enter the scenario ID
scenario_id = input("Enter Scenario ID (e.g. R5-T60): ").strip().upper()
scenario_id = scenario_id.replace('"', "").replace("'", "")

match = re.fullmatch(r"R(\d+)-T(\d+)", scenario_id)

if not match:
    raise ValueError("Use format R1-T70")

rank = int(match.group(1))
threshold = int(match.group(2))

# Load workbook
wb = load_workbook(io.BytesIO(file_bytes), data_only=True)
ws = wb[wb.sheetnames[0]]

# Detect intervention columns and total column dynamically
header_row = 1
headers = [ws.cell(header_row, col).value for col in range(1, ws.max_column + 1)]

# Find the "Total values" column
try:
    total_col = headers.index("Total values") + 1   # +1 because Excel columns are 1-based
except ValueError:
    raise ValueError("'Total values' column not found in row 1")

# Intervention columns sit between column B and the total values column.
intervention_start_col = 2
intervention_end_col = total_col - 1

intervention_headers = [
    ws.cell(header_row, col).value
    for col in range(intervention_start_col, intervention_end_col + 1)
]

# Extract totals from the total values column
def get_total(label):
    for r in range(1, ws.max_row + 1):
        if ws.cell(r, 1).value == label:
            return ws.cell(r, total_col).value
    return None

outputs = get_total("Number of opportunities")
area = get_total("Area")
storage = get_total("Storage")
carbon = get_total("Net carbon sequestration")
habitat = get_total("Net habitat units")
cost = get_total("Cost")

# Identify dominant intervention
counts = None
for r in range(1, ws.max_row + 1):
    if ws.cell(r, 1).value == "Number of opportunities":
        counts = [
            ws.cell(r, col).value if ws.cell(r, col).value is not None else 0
            for col in range(intervention_start_col, intervention_end_col + 1)
        ]
        break

dominant = None
if counts:
    s = pd.to_numeric(pd.Series(counts), errors="coerce").fillna(0)
    dominant = intervention_headers[s.idxmax()]

# Convert extracted values to numeric form
outputs = pd.to_numeric(outputs, errors="coerce")
area = pd.to_numeric(area, errors="coerce")
storage = pd.to_numeric(storage, errors="coerce")
carbon = pd.to_numeric(carbon, errors="coerce")
habitat = pd.to_numeric(habitat, errors="coerce")
cost = pd.to_numeric(cost, errors="coerce")

# Constants for Wansbeck baseline comparison
# baseline understood as r1-ss70 for my study
catchment_area = 333700000  # Wansbeck in m2
baseline_outputs = 3595     # R1-T70 (update if needed)
baseline_storage = 1195337.5
baseline_cost = 18604067.5

# Calculate derived fields
pct_cover = (area / catchment_area * 100) if pd.notna(area) else None

pct_outputs = ((outputs - baseline_outputs) / baseline_outputs * 100) if pd.notna(outputs) else None
pct_storage = ((storage - baseline_storage) / baseline_storage * 100) if pd.notna(storage) else None
pct_cost = ((cost - baseline_cost) / baseline_cost * 100) if pd.notna(cost) else None

# Create output row
row = [
    scenario_id,
    rank,
    threshold,
    outputs,
    area,
    pct_cover,
    storage,
    carbon,
    habitat,
    cost,
    dominant,
    pct_outputs,
    pct_storage,
    pct_cost,
    ""
]

columns = [
    "Scenario ID", "Rank", "Threshold", "Total Outputs", "Total Area (m²)",
    "% Catchment Covered", "Total Storage (m³)", "Total Net Carbon",
    "Total Net Habitat", "Total Cost (£)", "Dominant Intervention",
    "% Change Outputs (vs baseline)", "% Change Storage", "% Change Cost", "Notes"
]

df = pd.DataFrame([row], columns=columns)

display(df)

# Print paste-ready row for manual Excel entry
print("\nCOPY THIS INTO EXCEL:\n")
print("\t".join([str(x) if pd.notna(x) else "" for x in row]))

## 11. Optional sensitivity curve

This plotting section visualises how total recommended interventions respond to suitability score threshold changes. It assumes that `ss_scores`, `interventions`, and `inflection_ss` have already been defined in earlier analysis.

The output shows where the estimated point of inflection would occur, whereby the suitability score being increased, begings to have a redundant influence on the number of recomendations output. To note, these were measured in 5 point intervals, so there is some discrepency in the accuracy.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

fig, ax1 = plt.subplots(figsize=(12, 6))

# Background style
fig.patch.set_facecolor('white')
ax1.set_facecolor('#f9f9f9')

# Main line and fill
ax1.plot(ss_scores, interventions, 'o-', color='#2d5a27',
         linewidth=2.5, markersize=9, zorder=4,
         label='Total Recommended Interventions')
ax1.fill_between(ss_scores, interventions, alpha=0.12,
                 color='#2d5a27', zorder=2)

# Default SS70 line
ax1.axvline(x=70, color='#2980b9', linestyle=':', linewidth=2,
            zorder=3, label='Default Threshold (SS70)')

# Inflection point
ax1.axvline(x=inflection_ss, color='#c0392b', linestyle='--',
            linewidth=2, zorder=3,
            label=f'Steepest Rate of Change (~SS{int(inflection_ss)})')
inflection_y = np.interp(inflection_ss, ss_scores, interventions)
ax1.scatter([inflection_ss], [inflection_y],
            color='#c0392b', s=160, zorder=6, edgecolors='white', linewidth=1.5)

# Annotations
ax1.annotate('Default SS70\n(NI-recommended)',
             xy=(70, np.interp(70, ss_scores, interventions)),
             xytext=(66, max(interventions) * 0.55),
             fontsize=9.5, color='#2980b9', ha='center',
             arrowprops=dict(arrowstyle='->', color='#2980b9', lw=1.2))

ax1.annotate(f'Inflection ~SS{int(inflection_ss)}\n(Sensitivity peak)',
             xy=(inflection_ss, inflection_y),
             xytext=(inflection_ss + 2, inflection_y + max(interventions) * 0.12),
             fontsize=9.5, color='#c0392b', ha='left',
             arrowprops=dict(arrowstyle='->', color='#c0392b', lw=1.2))

# Gridlines
ax1.grid(True, which='major', linestyle='--', linewidth=0.6,
         color='#cccccc', alpha=0.8, zorder=1)
ax1.set_axisbelow(True)

# Axis labels and ticks
ax1.set_xlabel('Suitability Score Threshold', fontsize=12, labelpad=8)
ax1.set_ylabel('Total Recommended Interventions', fontsize=12, labelpad=8)
ax1.set_xticks(ss_scores)
ax1.tick_params(axis='both', labelsize=10)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'{int(x):,}'))

# Axis limits with padding
ax1.set_xlim(min(ss_scores) - 2, max(ss_scores) + 2)
ax1.set_ylim(0, max(interventions) * 1.12)

# Spine styling
for spine in ['top', 'right']:
    ax1.spines[spine].set_visible(False)
for spine in ['bottom', 'left']:
    ax1.spines[spine].set_color('#aaaaaa')

# Title and subtitle
fig.suptitle(
    'NatureInsight Output Sensitivity to Suitability Score Threshold',
    fontsize=13, fontweight='bold', x=0.13, ha='left', y=0.98)

fig.text(0.13, 0.93,
         'Wansbeck Catchment · Rank 1 · Total recommended interventions across SS50–SS80',
         fontsize=9.5, color='#555555', style='italic')

# Layout
plt.subplots_adjust(top=0.88, bottom=0.12, left=0.1, right=0.95)

# Legend
ax1.legend(fontsize=10, framealpha=0.9, loc='upper right',
           edgecolor='#cccccc')

# Source note
fig.text(0.13, 0.01,
         'Source: NatureInsight® (ARUP|SCALGO, 2024). Analysis: Author.',
         fontsize=8.5, color='#888888')

plt.tight_layout(rect=[0, 0.03, 1, 0.97])
plt.savefig('sensitivity_curve_polished.png', dpi=300, bbox_inches='tight')
print("Saved: sensitivity_curve_polished.png")
files.download('sensitivity_curve_polished.png')
plt.show()